#Celda 1: Instalación de Librerías

In [ ]:
!pip install -q mlflow xgboost pyngrok scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Celda 2: Carga y Preparación de Datos (Prevención de Overfitting)
En esta celda cargamos el dataset limpio vehicles_clean.csv, preprocesamos las variables numéricas y categóricas, y realizamos la separación train_test_split.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Cargar el dataset procesado
file_path = 'vehicles_clean.csv'
df = pd.read_csv(file_path, on_bad_lines='skip', engine='python')

# Selección de características (features) relevantes y variable objetivo (target)
features = ['year', 'odometer', 'manufacturer', 'fuel', 'transmission', 'drive', 'type']
target = 'price'

# Filtrar solo las columnas seleccionadas y eliminar filas sin target
df_ml = df[features + [target]].dropna(subset=[target])

X = df_ml[features]
y = df_ml[target]

# Identificar columnas por tipo
num_cols = ['year', 'odometer']
cat_cols = ['manufacturer', 'fuel', 'transmission', 'drive', 'type']

# Preprocesamiento: Imputación + Encoding
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# 2. SEPARACIÓN EN TRAIN Y TEST (Prevención de Overfitting)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset cargado:")
print(f"Muestras de entrenamiento (Train): {X_train.shape[0]}")
print(f"Muestras de prueba (Test): {X_test.shape[0]}")

Dataset cargado:
Muestras de entrenamiento (Train): 47613
Muestras de prueba (Test): 11904


#Celda 3: Configuración del Experimento en MLflow


In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Definir el nombre del experimento
mlflow.set_experiment("ValorAuto_Model_Comparison")

# Función auxiliar para calcular métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

print("MLflow preparado correctamente.")

2026/09/16 23:52:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/16 23:52:52 INFO mlflow.store.db.utils: Updating database tables
2026/09/16 23:52:55 INFO mlflow.tracking.fluent: Experiment with name 'ValorAuto_Model_Comparison' does not exist. Creating a new experiment.


MLflow preparado correctamente.


#Celda 4: Entrenamiento y Registro de Modelos en MLflow
Aquí entrenamos y registramos el Baseline, el Random Forest y el XGBoost.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Diccionario con los modelos a entrenar
models = {
    "Baseline_LinearRegression": {
        "model": LinearRegression(),
        "params": {}
    },
    "Model_RandomForest": {
        "model": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
        "params": {"n_estimators": 100, "max_depth": 15, "random_state": 42}
    },
    "Model_XGBoost": {
        "model": XGBRegressor(n_estimators=100, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1),
        "params": {"n_estimators": 100, "max_depth": 8, "learning_rate": 0.1, "random_state": 42}
    }
}

# Bucle para entrenar, predecir y registrar en MLflow
for model_name, config in models.items():
    with mlflow.start_run(run_name=model_name):
        print(f"Entrenando {model_name}...")

        # Crear pipeline completo
        clf = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', config["model"])])

        # Entrenamiento
        clf.fit(X_train, y_train)

        # Predicciones sobre el conjunto de prueba (Test)
        y_pred = clf.predict(X_test)

        # Métricas
        rmse, mae, r2 = eval_metrics(y_test, y_pred)

        # Log de Parámetros
        mlflow.log_params(config["params"])

        # Log de Métricas
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        # Log del Modelo usando cloudpickle para evitar bloqueos de skops
        mlflow.sklearn.log_model(
            sk_model=clf,
            name="model",
            serialization_format="cloudpickle"
        )

        print(f"[{model_name}] -> RMSE: {rmse:.2f} | MAE: {mae:.2f} | R2: {r2:.4f}\n")

Entrenando Baseline_LinearRegression...


2026/09/16 23:57:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Baseline_LinearRegression] -> RMSE: 8256.30 | MAE: 5670.19 | R2: 0.6527

Entrenando Model_RandomForest...


2026/09/16 23:58:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Model_RandomForest] -> RMSE: 6013.20 | MAE: 3713.92 | R2: 0.8158

Entrenando Model_XGBoost...


2026/09/16 23:58:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Model_XGBoost] -> RMSE: 6177.03 | MAE: 3905.75 | R2: 0.8056



#Celda 5: Generar Interfaz Gráfica (MLflow UI) para Ver la Tabla de Comparación
Para visualizar la interfaz de MLflow y tomar la evidencia de la comparación:

In [ ]:
import getpass
from pyngrok import ngrok

# 1. Iniciar MLflow en host 0.0.0.0
get_ipython().system_raw("mlflow ui --host 0.0.0.0 --port 5000 &")

# 2. Configurar Authtoken
ngrok.set_auth_token(getpass.getpass("Ingresa tu NGROK Authtoken: "))

# 3. Forzar reescritura de host para evitar 'Invalid Host header'
public_url = ngrok.connect(5000, host_header="rewrite")

print(f"\n👉 Abre tu interfaz de MLflow aquí: {public_url}")

Ingresa tu NGROK Authtoken: ··········

👉 Abre tu interfaz de MLflow aquí: NgrokTunnel: "https://grazing-reformist-swimsuit.ngrok-free.dev" -> "http://localhost:5000"


Para cumplir con esta historia de usuario debes ejecutar dos pasos: primero, registrar el mejor modelo entrenado en el MLflow Model Registry bajo la etapa Production, y segundo, crear el script en src/pipeline/ para realizar la inferencia batch sobre las combinaciones.

#Registrar el modelo en el Model Registry (Etapa Production)
Ejecuta esta celda en Google Colab para promover automáticamente tu mejor modelo (XGBoost o Random Forest con menor RMSE) al registro oficial de MLflow.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# 1. Obtener el mejor run del experimento ordenado por RMSE
experiment = mlflow.get_experiment_by_name("ValorAuto_Model_Comparison")
best_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"]
).iloc[0]

run_id = best_run.run_id
model_uri = f"runs:/{run_id}/model"
model_name = "ValorAuto_Model"

# 2. Registrar el modelo en el Registry
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

# 3. Asignar la etapa "Production" al modelo
client.transition_model_version_stage(
    name=model_name,
    version=registered_model.version,
    stage="Production"
)

print(f"Modelo versión {registered_model.version} asignado a Production en MLflow Registry.")

Successfully registered model 'ValorAuto_Model'.
2026/09/17 00:29:05 WARNING mlflow.tracking._model_registry.fluent: Run with id 0a42168477d443f3b7157cdc07345143 has no artifacts at artifact path 'model', registering model based on models:/m-02276b5de78b4f0589cd367706dbdd21 instead


Modelo versión 1 asignado a Production en MLflow Registry.


Created version '1' of model 'ValorAuto_Model'.
/tmp/ipykernel_489/891875463.py:21: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


#Crear el script de precálculo (src/pipeline/predict_combinations.py)
Esta celda crea directamente el archivo .py en la estructura de tu proyecto (src/pipeline/). Utiliza inferencia por lotes (vectorizada) sobre el DataFrame entero para mitigar el riesgo de tiempo de cómputo excesivo.

In [ ]:
import os

# Crear la estructura de carpetas src/pipeline/
os.makedirs('src/pipeline', exist_ok=True)

script_content = """import os
import pandas as pd
import mlflow.pyfunc

def main():
    # 1. Cargar el modelo directamente desde MLflow Model Registry (Stage: Production)
    model_name = "ValorAuto_Model"
    stage = "Production"
    model_uri = f"models:/{model_name}/{stage}"

    print(f"Cargando modelo desde MLflow Model Registry: {model_uri}...")
    model = mlflow.pyfunc.load_model(model_uri)

    # 2. Cargar combinaciones/dataset completo
    input_path = 'data/processed/vehicles_clean.csv'
    output_path = 'data/processed/predicted_prices.csv'

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"No se encontró el archivo de origen en {input_path}")

    df = pd.read_csv(input_path)

    # Columnas esperadas por el pipeline
    features = ['year', 'odometer', 'manufacturer', 'fuel', 'transmission', 'drive', 'type']
    X_combinations = df[features]

    # 3. Inferencia Batch Vectorizada (Evita bucles for para máxima eficiencia)
    print(f"Calculando precios para {len(X_combinations)} combinaciones...")
    predictions = model.predict(X_combinations)

    # 4. Generar tabla de combinación -> precio predicho
    df_result = X_combinations.copy()
    df_result['predicted_price'] = predictions.round(2)

    # 5. Guardar el archivo final
    os.makedirs('data/processed', exist_ok=True)
    df_result.to_csv(output_path, index=False)
    print(f"Proceso finalizado con éxito. Resultado guardado en: {output_path}")

if __name__ == "__main__":
    main()
"""

# Guardar el script en src/pipeline/predict_combinations.py
with open('src/pipeline/predict_combinations.py', 'w') as f:
    f.write(script_content)

print("Script creado exitosamente en src/pipeline/predict_combinations.py")

Script creado exitosamente en src/pipeline/predict_combinations.py


In [ ]:
import os
import shutil

# Crear la estructura de carpetas y mover el archivo
os.makedirs('data/processed', exist_ok=True)
if os.path.exists('vehicles_clean.csv'):
    shutil.move('vehicles_clean.csv', 'data/processed/vehicles_clean.csv')
    print("✅ Archivo movido exitosamente a data/processed/vehicles_clean.csv")
else:
    print("El archivo ya se encuentra en su ubicación o no está en la raíz.")

✅ Archivo movido exitosamente a data/processed/vehicles_clean.csv


In [ ]:
!python src/pipeline/predict_combinations.py

Cargando modelo desde MLflow Model Registry: models:/ValorAuto_Model/Production...
Calculando precios para 100625 combinaciones...
Proceso finalizado con éxito. Resultado guardado en: data/processed/predicted_prices.csv
